<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-12/AI_Modul_12.3_Praktikum_Pooling.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 12.3: Praktikum Pooling (Max, Average, dan Global Average Pooling)
**Tujuan Pembelajaran:**
1. Membangun operasi Max Pooling dan Average Pooling secara matematis manual (NumPy).
2. Menguji sifat invariansi translasi spasial pada model dengan vs tanpa pooling.
3. Mengimplementasikan Global Average Pooling (GAP) di PyTorch dan menganalisis reduksi parameter.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch Versi: {torch.__version__}")

### Bagian 1: Verifikasi Numerik NumPy vs PyTorch Pooling

In [ ]:
def manual_max_pool(x, size=2, stride=2):
    H, W = x.shape
    H_out = (H - size) // stride + 1
    W_out = (W - size) // stride + 1
    out = np.zeros((H_out, W_out), dtype=x.dtype)
    for i in range(H_out):
        for j in range(W_out):
            out[i, j] = np.max(x[i*stride:i*stride+size, j*stride:j*stride+size])
    return out

# Matriks sampel 4x4
matrix_sample = np.array([
    [10, 20, 15,  5],
    [ 8, 35, 12,  4],
    [ 2, 14, 40, 18],
    [ 9, 11, 22, 30]
], dtype=np.float32)

np_res = manual_max_pool(matrix_sample, size=2, stride=2)

# PyTorch MaxPool2d
t_in = torch.tensor(matrix_sample).unsqueeze(0).unsqueeze(0)
pool_pt = nn.MaxPool2d(2, 2)
pt_res = pool_pt(t_in).squeeze().numpy()

print("Hasil Manual NumPy:")
print(np_res)
print("Hasil PyTorch:")
print(pt_res)
assert np.allclose(np_res, pt_res), "Hasil pooling berbeda!"
print("[VALIDASI SUKSES] Logika Max Pooling terverifikasi identik!")

### Bagian 2: Eksperimen Invariansi Translasi Citra Tajuk Sawit

In [ ]:
# Buat representasi sintetis tajuk sawit (titik terang di tengah)
palm_orig = np.zeros((1, 1, 16, 16), dtype=np.float32)
palm_orig[0, 0, 7:9, 7:9] = 1.0 # Inti tajuk di pusat

# Geser (shift) 2 piksel ke kanan bawah
palm_shifted = np.zeros((1, 1, 16, 16), dtype=np.float32)
palm_shifted[0, 0, 9:11, 9:11] = 1.0

# Uji dengan MaxPool2d(4, 4)
pool_test = nn.MaxPool2d(kernel_size=4, stride=4)
out_orig = pool_test(torch.tensor(palm_orig))
out_shifted = pool_test(torch.tensor(palm_shifted))

# Hitung kesamaan representasi pooled
cos_sim = torch.cosine_similarity(out_orig.flatten(), out_shifted.flatten(), dim=0)
print(f"Cosine Similarity Setelah Max Pooling: {cos_sim.item():.4f}")
print("[ANALISIS] Nilai aktivasi puncak tetap terdeteksi pada region spasial yang konsisten!")

### Bagian 3: Efisiensi Global Average Pooling (GAP)

In [ ]:
# Model dengan Flatten + Dense Besar (Pendekatan VGG)
class VGGStyleHead(nn.Module):
    def __init__(self, in_channels=64, num_classes=5):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_channels * 8 * 8, 512),
            nn.ReLU(),
            nn.Linear(512, num_classes)
        )
    def forward(self, x):
        return self.fc(x)

# Model dengan Global Average Pooling (Pendekatan ResNet)
class GAPStyleHead(nn.Module):
    def __init__(self, in_channels=64, num_classes=5):
        super().__init__()
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(in_channels, num_classes)
    def forward(self, x):
        return self.fc(torch.flatten(self.gap(x), 1))

head_vgg = VGGStyleHead()
head_gap = GAPStyleHead()

p_vgg = sum(p.numel() for p in head_vgg.parameters())
p_gap = sum(p.numel() for p in head_gap.parameters())

print(f"Parameter VGG-Style Head (Dense): {p_vgg:,}")
print(f"Parameter GAP-Style Head (GAP):   {p_gap:,}")
print(f"Rasio Kompresi Parameter: GAP menggunakan hanya {p_gap / p_vgg * 100:.2f}% parameter dari Dense!")

# Visualisasi komparasi parameter
plt.figure(figsize=(6, 4))
plt.bar(['Dense Head', 'GAP Head'], [p_vgg, p_gap], color=['#D32F2F', '#388E3C'])
plt.title('Perbandingan Jumlah Parameter Classifier Head')
plt.ylabel('Jumlah Parameter (Skala Linier)')
plt.yscale('log')
plt.ylabel('Jumlah Parameter (Skala Logaritmik)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig('praktikum_gap_parameter_compression_12_3.png', dpi=200)
plt.close()
print("[OK] Grafik perbandingan disimpan sebagai 'praktikum_gap_parameter_compression_12_3.png'")